## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-06-10 09:32:14--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  3.39MB/s    in 0.3s    

2025-06-10 09:32:15 (3.39 MB/s) - ‘input.txt’ saved [1115394/1115394]



## Step1. Reading and exploring data

In [7]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [8]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [9]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [10]:
# here are all the unique characters that occur in this text
# 在資料集中所有的字元 (characters) 種類
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


## Step2. Tokenization, train/val split

### Tokenization（標記化）與數據編碼

目的: 將人類可讀的文字轉換為機器學習模型可以處理的數字序列，並準備進行模型訓練

> **核心概念**
> - **機器學習模型只能處理數字**，不能直接處理文字
> - 需要建立文字和數字之間的雙向映射關係

--

**實作步驟**

* Step 1: 建立映射關係
    1. 建立映射字典
    2. 定義編碼/解碼函數

* Step 2: 數據集編碼與張量轉換
    1. 編碼整個數據集
    2. 檢查數據格式

--

> 特點 & 重要性
> - **字符級別** tokenization：每個字符對應一個數字
> - 簡單直接，但效率不如現代方法（**SentencePiece**、**tiktoken**）
> - 建立統一的數據結構，方便批次處理和訓練/驗證集分割
> 
> **對 GPT 來說**，原本的文字現在看起來就是這些數字序列
> 這是整個 GPT 訓練流程的基礎步驟，沒有 tokenization 就無法將文字數據餵給神經網路進行學習。



In [22]:
# create a mapping from characters to integers
# 1. 建立映射字典
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# 2. 定義編碼/解碼函數
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers.
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string.
# encoder: 字串 => 數字
# decoder: 數字 => 字串
# https://github.com/google/sentencepiece / https://github.com/openai/tiktoken

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
# 1. 編碼整個數據集
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)

# 2. 檢查數據格式
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

### Train/val split 數據分割與批次處理

1. 方法1: 訓練/驗證集分割

    將整個數據集按 9:1 的比例分割為訓練集和驗證集，這是機器學習中的標準做法

2. 方法2: 序列生成與上下文窗口 (Context Window)

    GPT 模型的核心概念是**上下文窗口** (Context Window)，即模型在預測下一個token時能夠"看到"的前面token的數量。

    **重要概念**：
    - 每個位置的輸入都對應一個預測目標
    - 模型學習在給定前文的情況下預測下一個token
    - 這種設計讓我們能從一個序列中獲得多個訓練樣本

3. 方法3: 批次處理 (Batch Processing)

    為了提高訓練效率，我們將多個序列組合成批次進行並行處理：

    ```python
    def get_batch(split):
        # 生成一個小批次的輸入x和目標y
        data = train_data if split == 'train' else val_data
        ix = torch.randint(len(data) - block_size, (batch_size,))  # 隨機選擇起始位置
        x = torch.stack([data[i:i+block_size] for i in ix])        # 輸入序列
        y = torch.stack([data[i+1:i+block_size+1] for i in ix])    # 目標序列
        return x, y
    ```

    **維度說明**：
    - `batch_size = 4`: 每次處理4個獨立的序列
    - `block_size = 8`: 每個序列長度為8個token
    - 最終形狀：`x.shape = (4, 8)`, `y.shape = (4, 8)`

    **訓練樣本生成**：
    每個batch實際上包含了 `batch_size × block_size = 32` 個訓練樣本，因為每個位置都是一個獨立的預測任務。

    > **關鍵洞察**：這種設計讓GPT能夠高效地學習語言模式，每次前向傳播都能從多個上下文長度中學習，從而提高訓練效率。


In [33]:
# Let's now split up the data into train and validation sets
# 數據分割
n = int(0.9*len(data)) # 前90%作為訓練集，剩餘10%作為驗證集
train_data = data[:n]
val_data = data[n:]

# 檢查數據格式
print(f"train has {len(train_data)} examples, val has {len(val_data)} examples")

train has 1003854 examples, val has 111540 examples


In [35]:
# 序列生成與上下文窗口 (Context Window)
block_size = 8 # 上下文長度，即模型能看到的最大前文長度

# 生成上下文窗口
x = train_data[:block_size] # 輸入序列: [18, 47, 56, 57, 58, 1, 15, 47]
y = train_data[1:block_size+1] # 目標序列: [47, 56, 57, 58, 1, 15, 47, 58]

# 檢查上下文窗口
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")


when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [39]:
torch.manual_seed(1337)

# 批次處理 (Batch Processing)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # 生成一個小批次的輸入 x 和目標 y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

# 檢查小批次輸入和目標
for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [38]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


## Simplest baseline: bigram language model, loss, generation

### BigramLanguageModel 實作

這是一個簡單的 Bigram 語言模型實作，作為 GPT 架構的基礎版本。

#### 模型架構概述

**Bigram 模型**是最簡單的語言模型，它僅基於當前 token 來預測下一個 token，不考慮更長的上下文。

1. 模型初始化

    - `nn.Embedding(vocab_size, vocab_size)` 創建一個查找表
    - 每個輸入 token 直接對應到一個向量，該向量表示所有 vocab 中 token 的 logits
    - 這是最簡單的實現：當前 token → 下一個 token 的機率

2. 前向傳播 (Forward Pass)

    **處理流程**：
    1. **Embedding 查找**：每個 token 索引 → 對應的 logits 向量
    2. **張量重塑**：將 3D 張量展平為 2D，以計算損失
    3. **損失計算**：使用交叉熵比較預測與真實目標

3. 文本生成 (Generation)

    **生成過程**：
    1. **預測**：基於當前序列預測下一個 token
    2. **機率化**：將 logits 轉換為機率分布
    3. **採樣**：隨機選擇下一個 token（保持多樣性）
    4. **擴展**：將新 token 加入序列
    5. **迭代**：重複上述過程

4. 模型測試與示例

#### 模型特點與限制

**優點**：
- 📚 簡單易懂，適合學習基礎概念
- ⚡ 計算效率高
- 🎯 清晰展示了語言模型的核心機制

**限制**：
- 🔒 只考慮前一個 token，無法捕捉長距離依賴
- 📉 生成品質有限，缺乏連貫性
- 🎲 無法理解更複雜的語言模式

> 💡 **下一步**：這個 Bigram 模型將逐步演化為完整的 Transformer 架構，加入 Self-Attention、多頭注意力、位置編碼等機制。

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    # 1. 模型初始化
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        # 核心組件：Token Embedding 表
        # 形狀: (vocab_size, vocab_size)
        # 每個 token 直接映射到所有可能下一個 token 的機率分布
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    # 2. 前向傳播 (Forward Pass)
    def forward(self, idx, targets=None):
        """
        Args:
            idx: 輸入序列 (B, T) - Batch size, Time steps
            targets: 目標序列 (B, T) - 用於計算損失
        
        Returns:
            logits: 預測機率 (B, T, C) - C 是 vocab_size
            loss: 交叉熵損失 (如果提供 targets)
        """
        # idx and targets are both (B,T) tensor of integers
        # Step 1: 獲取 logits
        logits = self.token_embedding_table(idx) # (B,T,C)

        # Step 2: 計算損失 (如果提供目標序列)
        if targets is None:
            loss = None
        else:
            # 重塑張量以符合 CrossEntropy 要求 (B*T, C)
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            # 計算交叉熵損失
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # 3. 文本生成 (Generation)
    def generate(self, idx, max_new_tokens):
        """
        自回歸生成新的 token 序列
        
        Args:
            idx: 起始上下文 (B, T)
            max_new_tokens: 要生成的新 token 數量
        
        Returns:
            idx: 擴展後的序列 (B, T + max_new_tokens)
        """
        for _ in range(max_new_tokens):
            # Step 1: 獲取當前序列的預測
            logits, loss = self(idx)
            
            # Step 2: 只使用最後一個時間步的預測
            logits = logits[:, -1, :]  # (B, C)
            
            # Step 3: 轉換為機率分布
            probs = F.softmax(logits, dim=-1)  # (B, C)
            
            # Step 4: 從分布中採樣下一個 token
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            
            # Step 5: 將新 token 附加到序列
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx

# 4. 模型測試與示例

# 建立並測試模型
m = BigramLanguageModel(vocab_size)  
logits, loss = m(xb, yb)

print(f"Logits shape: {logits.shape}")  # [batch_size*block_size, vocab_size]
print(f"Initial loss: {loss}")          # ~4.17 (理論上 -ln(1/65))

# 生成文本範例 (訓練前)
generated_text = decode(
    m.generate(
        idx=torch.zeros((1, 1), dtype=torch.long), 
        max_new_tokens=100
    )[0].tolist()
)
print(f"Generated text: {generated_text}")


Logits shape: torch.Size([32, 65])
Initial loss: 4.878634929656982
Generated text: 
SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


#### 模型訓練與文本生成

1. 訓練配置

    **訓練設置說明**：
    - **優化器**：AdamW（Adam 的改進版本，帶權重衰減）
    - **學習率**：1e-3（0.001）- 適中的學習率，平衡訓練速度與穩定性
    - **參數**：模型的所有可訓練參數（這裡只有 embedding table）

2. 訓練循環

    **訓練流程解析**：

    1. **數據採樣** 📊
        - 從訓練集中隨機採樣 `batch_size=32` 個序列
        - 每個序列長度為 `block_size=8`

    2. **前向傳播** ➡️
        - 計算模型預測 logits
        - 計算交叉熵損失

    3. **梯度處理** 🔄
        - `zero_grad()`: 清除上一步的梯度
        - `set_to_none=True`: 更高效的記憶體使用

    4. **反向傳播** ⬅️
        - `loss.backward()`: 計算梯度

    5. **參數更新** 🔧
        - `optimizer.step()`: 根據梯度更新模型參數

3. 損失變化趨勢

    **損失解釋**：
    - **初始損失 4.17**：未訓練模型的隨機預測，等於 `-ln(1/vocab_size)`
    - **損失下降**：表明模型正在學習數據中的字符模式
    - **目標**：損失越低，模型預測越準確

4. 文本生成結果

    訓練前的生成（隨機輸出） -> 訓練後的生成

    **生成參數說明**：
    - **起始序列**：`torch.zeros((1, 1))` - 從索引0（通常是換行符）開始
    - **生成長度**：500個新字符
    - **採樣方式**：機率採樣（保持文本多樣性）


5. 訓練效果分析

    **改進指標**：
    - ✅ **損失顯著下降**：從 4.17 → 2.5
    - ✅ **字符分布學習**：模型學會了更合理的字符組合
    - ✅ **基本模式識別**：能夠產生類似英文的字符序列

    **仍存在的限制**：
    - ❌ **語義連貫性**：缺乏長距離語義理解
    - ❌ **語法結構**：無法形成正確的語法結構
    - ❌ **上下文記憶**：只能記住前一個字符的信息

In [42]:
# create a PyTorch optimizer
# 創建 PyTorch 優化器
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [55]:
# 設置批次大小
batch_size = 32

# 訓練循環
for steps in range(1000):  # 增加步數以獲得更好的結果...
    # Step 1: 採樣一個批次的數據
    xb, yb = get_batch('train')

    # Step 2: 評估損失
    logits, loss = m(xb, yb)
    
    # Step 3: 梯度清零
    optimizer.zero_grad(set_to_none=True)
    
    # Step 4: 反向傳播
    loss.backward()
    
    # Step 5: 參數更新
    optimizer.step()


# 訓練結果
print(loss.item())


2.3699088096618652


In [59]:
# 文本生成結果
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


ARWhace toindilithespr alily well fthis I as r ovemyorerouthis matofrhe lavefessuladvoms tod:3BR: y Ind igng ies we:
Car Whan:
Hpout.
Thont fo ind ove l ld:
TE llaclarr.
en th.
WAD win;
WI.
AMyothomo c
AUSI bend ant frrthi's buthareve y s avel,'t,
K: f-m roke sth f tid Ch IUpechadd angoutee, winmerthorouow-wsqus,
MI whines, rcothheanoneshe olithel CLIIFrdatit t be w yongayoy.


Wharoputhe steaple abed
DWhin.
Clang-
QUNGRDoome dir hay modo
ILISust h t ar oPORETwe atino g;
P UE:
Whyofa,
RWeallly s


## The mathematical trick in self-attention

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [ ]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [ ]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [ ]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [ ]:
k.var()

tensor(1.0449)

In [ ]:
q.var()

tensor(1.0700)

In [ ]:
wei.var()

tensor(1.0918)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [ ]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [ ]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [ ]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



### Full finished code, for reference

You may want to refer directly to the git repo instead though.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5090, val loss 2.5058
step 300: train loss 2.4198, val loss 2.4340
step 400: train loss 2.3503, val loss 2.3567
step 500: train loss 2.2970, val loss 2.3136
step 600: train loss 2.2410, val loss 2.2506
step 700: train loss 2.2062, val loss 2.2198
step 800: train loss 2.1638, val loss 2.1871
step 900: train loss 2.1232, val loss 2.1494
step 1000: train loss 2.1020, val loss 2.1293
step 1100: train loss 2.0704, val loss 2.1196
step 1200: train loss 2.0382, val loss 2.0798
step 1300: train loss 2.0249, val loss 2.0640
step 1400: train loss 1.9922, val loss 2.0354
step 1500: train loss 1.9707, val loss 2.0308
step 1600: train loss 1.9614, val loss 2.0474
step 1700: train loss 1.9393, val loss 2.0130
step 1800: train loss 1.9070, val loss 1.9943
step 1900: train loss 1.9057, val loss 1.9871
step 2000: train loss 1.8834, val loss 1.9954
step 2100: train loss 1.

In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary

# Assuming 'm' is your BigramLanguageModel instance
summary(m, input_size=(batch_size, block_size), dtypes=[torch.long, torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
BigramLanguageModel                      [4, 8, 65]                --
├─Embedding: 1-1                         [4, 8, 65]                4,225
Total params: 4,225
Trainable params: 4,225
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.02
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 0.02
Estimated Total Size (MB): 0.03